In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("online_retail_cleaned.csv")
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
print(df.shape)

(805549, 9)


In [3]:
reference_date =  df["invoicedate"].max() + pd.Timedelta(days=1)
print("Refernce Date:", reference_date)

Refernce Date: 2011-12-10 12:50:00


In [5]:
rfm = df.groupby("customer_id").agg(
    recency   = ("invoicedate", lambda x: (reference_date - x.max()).days),
    frequency = ("invoice", "nunique"),
    monetary  = ("totalprice", "sum")
).reset_index()

print(rfm.shape)
rfm.head(10)

(5878, 4)


,customer_id,recency,frequency,monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40
5,12351.0,375,1,300.93
6,12352.0,36,10,2849.84
7,12353.0,204,2,406.76
8,12354.0,232,1,1079.40
9,12355.0,214,2,947.61


In [6]:
rfm["r_score"] = pd.qcut(rfm["recency"], q=5, labels=[5,4,3,2,1])
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1,2,3,4,5])
rfm["m_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1,2,3,4,5])

rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)
print(rfm.head(10))

   customer_id  recency  frequency  monetary r_score f_score m_score rfm_score
0      12346.0      326         12  77556.46       2       5       5       255
1      12347.0        2          8   5633.32       5       4       5       545
2      12348.0       75          5   2019.40       3       4       4       344
3      12349.0       19          4   4428.69       5       3       5       535
4      12350.0      310          1    334.40       2       1       2       212
5      12351.0      375          1    300.93       2       1       2       212
6      12352.0       36         10   2849.84       4       5       4       454
7      12353.0      204          2    406.76       2       2       2       222
8      12354.0      232          1   1079.40       2       1       3       213
9      12355.0      214          2    947.61       2       2       3       223


In [7]:
def assign_segment(row):
    r = int(row["r_score"])
    f = int(row["f_score"])
    m = int(row["m_score"])
    
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r >= 3 and f <= 2:
        return "Potential Loyalists"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2 and m >= 3:
        return "Cant Lose Them"
    else:
        return "Lost"

rfm["segment"] = rfm.apply(assign_segment, axis=1)
print(rfm["segment"].value_counts())

segment
Loyal Customers        1403
Champions              1300
Lost                   1275
At Risk                 824
New Customers           443
Potential Loyalists     385
Cant Lose Them          248
Name: count, dtype: int64


In [8]:
rfm.to_csv("rfm_scores.csv", index=False)
print("Saved!")

Saved!
